Testing connection 

In [1]:
from pymongo import MongoClient

def connect_to_database():
    try:
        client = MongoClient("mongodb://db:27017/", serverSelectionTimeoutMS=5000)
        db = client.fingerprintDB
        # Teste die Verbindung
        db.command("ping")
        print("Connected to the database.")
        return db
    except Exception as e:
        print(f"Error connecting to MongoDB server: {e}")
        return None

db = connect_to_database()

Connected to the database.


In [1]:
import pymongo
import pandas as pd
import matplotlib.pyplot as plt
import base64
from PIL import Image
from io import BytesIO

# Funktion zur Herstellung der Verbindung zur MongoDB-Datenbank
def connect_to_database():
    try:
        client = pymongo.MongoClient('mongodb://localhost:27018/')
        db = client['fingerprintDB']
        print("Connected to the database.")
        return db
    except pymongo.errors.ConnectionError as err:
        print(f"Error: {err}")
        return None

# Funktion zum Abrufen der Benutzer aus der Datenbank
def fetch_users_from_database(db):
    try:
        print("Fetching users from the database...")
        filter = {}
        project = {
            'username': 1,
            '_id': 0
        }
        users = db.fingerprints.find(filter=filter, projection=project)
        user_list = [user["username"] for user in users]
        print(f"Retrieved {len(user_list)} users: {user_list}")
        return user_list
    except Exception as e:
        print(f"Error fetching users: {e}")
        return []

# Funktion zum Abrufen von Fingerabdrücken für einen Benutzer
def fetch_fingerprints_for_user(db, username):
    try:
        print(f"Fetching fingerprints for user: {username}")
        user = db.fingerprints.find_one({"username": username}, {"_id": 1})
        if not user:
            print(f"No user found with username: {username}")
            return []
        
        fingerprint_id = user.get("_id")
        print(f"Retrieved fingerprint ID for user: {username}: {fingerprint_id}")
        return fingerprint_id
    except Exception as e:
        print(f"Error fetching fingerprint ID: {e}")
        return []

# Funktion zum Abrufen von Fingerabdruckdaten basierend auf ID
def fetch_fingerprint_data(db, fingerprint_id):
    try:
        if not fingerprint_id:
            print("No fingerprint ID provided.")
            return []
        print(f"Fetching fingerprint data for ID: {fingerprint_id}")
        fingerprints = db.canvassamples.find({"fingerprintId": fingerprint_id}, {"sampleData": 1, "_id": 0}).limit(5)
        fingerprint_data = [fp["sampleData"] for fp in fingerprints]
        print(f"Retrieved {len(fingerprint_data)} fingerprint data entries.")
        return fingerprint_data
    except Exception as e:
        print(f"Error fetching fingerprint data: {e}")
        return []

# Funktion zum Dekodieren von Base64-Bildern
def decode_base64_image(base64_string):
    image_data = base64.b64decode(base64_string)
    image = Image.open(BytesIO(image_data))
    return image

# Verbindung zur Datenbank herstellen
db = connect_to_database()

# Daten abrufen und anzeigen
if db is not None:
    users = fetch_users_from_database(db)
    for username in users:
        fingerprint_id = fetch_fingerprints_for_user(db, username)
        if fingerprint_id:
            fingerprints = fetch_fingerprint_data(db, fingerprint_id)
            if fingerprints:
                print(f"Fingerprints for user: {username}")
                for fingerprint in fingerprints:
                    print(fingerprint)  # Ausgabe der Fingerabdrücke als Text in der Konsole
                fig, axes = plt.subplots(1, len(fingerprints), figsize=(15, 5))
                fig.suptitle(f'Fingerprints for user: {username}')
                for ax, fingerprint in zip(axes, fingerprints):
                    image = decode_base64_image(fingerprint)
                    ax.imshow(image)
                    ax.axis('off')
                plt.show()
            else:
                print(f"No fingerprints found for user: {username}")
        else:
            print(f"No fingerprint ID found for user: {username}")
else:
    print("Failed to connect to the database.")

Connected to the database.
Fetching users from the database...
Error fetching users: localhost:27018: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 67112417f66eb5570185a296, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27018) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27018: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>


Fehlende Werte überprüfen

In [8]:
from pymongo import MongoClient

# Verbindung zur MongoDB herstellen
client = MongoClient('mongodb://localhost:27018/')  # Ensure the correct port is used
try:
    # Check if the server is available
    client.admin.command('ping')
    print("Connected to MongoDB server.")
except Exception as e:
    print(f"Error connecting to MongoDB server: {e}")
client = MongoClient('mongodb://localhost:27018/')  # Ensure the correct port is used
db = client['deine_datenbank']
collection = db['deine_sammlung']

# Dokumente mit fehlendem 'fingerprintHash'-Feld finden
fehlende_fingerprintHash = collection.find({ "fingerprintHash": { "$exists": False } })

# Anzahl der Dokumente mit fehlendem 'fingerprintHash'-Feld ausgeben
print(f"Anzahl der Dokumente mit fehlendem 'fingerprintHash'-Feld: {collection.count_documents({ 'fingerprintHash': { '$exists': False } })}")

# Dokumente mit fehlendem 'username'-Feld finden
fehlende_username = collection.find({ "username": { "$exists": False } })

# Anzahl der Dokumente mit fehlendem 'username'-Feld ausgeben
print(f"Anzahl der Dokumente mit fehlendem 'username'-Feld: {collection.count_documents({ 'username': { '$exists': False } })}")


Error connecting to MongoDB server: localhost:27018: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 6710c181a6774bd160f8ff6f, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27018) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27018: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>


ServerSelectionTimeoutError: localhost:27018: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 6710c19fa6774bd160f8ff70, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27018) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27018: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>